# Bull Reentry BC Warm-start Strict Gate

This notebook composes the current strict retention filter with the bull reentry target contract. It is a training handoff only: no actor promotion or retention is allowed until the full fixed-split and walk-forward gate passes.

In [1]:
from pathlib import Path
import csv
import json
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
CARD = ROOT / 'runtime/model_selection/card/bull_reentry_bc_warmstart_gate_v1'
STRICT_PATH = ROOT / 'runtime/model_selection/card/strict_candidate_retention_filter_v1/summary.json'
TARGET_PATH = ROOT / 'runtime/model_selection/card/bull_reentry_joint_target_contract_v1/summary.json'

from src.soxl_quant.app.bc_warmstart_strict_gate import (
    derive_bc_warmstart_strict_gate,
    render_bc_warmstart_strict_gate,
)

strict_payload = json.loads(STRICT_PATH.read_text(encoding='utf-8'))
target_contract = json.loads(TARGET_PATH.read_text(encoding='utf-8'))
strict_summary = strict_payload.get('summary', strict_payload)
gate = derive_bc_warmstart_strict_gate(
    strict_candidate_filter=strict_summary,
    target_contract=target_contract,
)
gate

{'decision': 'bull_reentry_bc_warmstart_allowed_no_promotion',
 'bc_warm_start_allowed': True,
 'training_allowed_next': True,
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'ppo_allowed_next': False,
 'reference_baseline': 'soxl_backtest_policy_reimplementation_v1',
 'required_gates': {'required_wf_windows': ['wf_2020',
   'wf_2022',
   'wf_2023_2024'],
  'max_walkforward_drawdown_pct': 20.0,
  'reference_baseline': 'soxl_backtest_policy_reimplementation_v1',
  'required_best_return_pct': 213.143462},
 'required_next_gate': 'full_fixed_split_and_walkforward_retention_gate',
 'next_mechanism': 'wire_fold_local_bull_reentry_joint_targets_into_bc_warm_start_then_full_walkforward_gate',
 'source_target_contract': {'experiment': 'bull_reentry_joint_target_contract_v1',
  'decision': 'joint_target_contract_bull_reentry_gate_ready_for_training_handoff',
  'rows': 3,
  'bull_reentry_target_rows': 2,
  'eval_overlap_rows': 0,
  'risk_conflict_target_rows': 0},
 'strict_candid

## Current Gate State

The cell below records the authoritative current state used by the handoff.

In [2]:
current_state = {
    'strict_filter_decision': strict_summary.get('decision'),
    'passing_profile_count': strict_summary.get('passing_profile_count'),
    'promotable_profile_count': strict_summary.get('promotable_profile_count'),
    'actor_retention_allowed_count': strict_summary.get('actor_retention_allowed_count'),
    'reference_baseline': strict_summary.get('reference_baseline'),
    'target_contract_decision': target_contract.get('decision'),
    'bull_reentry_target_rows': target_contract.get('bull_reentry_target_rows'),
    'eval_overlap_rows': target_contract.get('eval_overlap_rows'),
    'risk_conflict_target_rows': target_contract.get('risk_conflict_target_rows'),
    'bc_warm_start_allowed': gate.get('bc_warm_start_allowed'),
    'promotion_allowed': gate.get('promotion_allowed'),
    'actor_retention_allowed': gate.get('actor_retention_allowed'),
}
current_state

{'strict_filter_decision': 'strict_candidate_retention_filter_no_promotable_candidates',
 'passing_profile_count': 0,
 'promotable_profile_count': 0,
 'actor_retention_allowed_count': 0,
 'reference_baseline': 'soxl_backtest_policy_reimplementation_v1',
 'target_contract_decision': 'joint_target_contract_bull_reentry_gate_ready_for_training_handoff',
 'bull_reentry_target_rows': 2,
 'eval_overlap_rows': 0,
 'risk_conflict_target_rows': 0,
 'bc_warm_start_allowed': True,
 'promotion_allowed': False,
 'actor_retention_allowed': False}

## Persist Sidecars

In [3]:
CARD.mkdir(parents=True, exist_ok=True)
summary = {
    'experiment': 'bull_reentry_bc_warmstart_gate_v1',
    **gate,
    'source_paths': {
        'strict_candidate_retention_filter': str(STRICT_PATH.relative_to(ROOT)),
        'bull_reentry_joint_target_contract': str(TARGET_PATH.relative_to(ROOT)),
    },
    'sidecars': {
        'summary_json': 'runtime/model_selection/card/bull_reentry_bc_warmstart_gate_v1/summary.json',
        'selection_json': 'runtime/model_selection/card/bull_reentry_bc_warmstart_gate_v1/selection.json',
        'markdown': 'runtime/model_selection/card/bull_reentry_bc_warmstart_gate_v1/bc_warmstart_strict_gate.md',
        'source_manifest_csv': 'runtime/model_selection/card/bull_reentry_bc_warmstart_gate_v1/source_manifest.csv',
    },
}
selection = {
    'experiment': summary['experiment'],
    'decision': gate['decision'],
    'training_allowed_next': gate['training_allowed_next'],
    'bc_warm_start_allowed': gate['bc_warm_start_allowed'],
    'promotion_allowed': gate['promotion_allowed'],
    'actor_retention_allowed': gate['actor_retention_allowed'],
    'required_next_gate': gate['required_next_gate'],
    'next_mechanism': gate['next_mechanism'],
    'reference_baseline': gate['reference_baseline'],
    'required_gates': gate['required_gates'],
}
source_manifest = [
    {
        'source_label': 'strict_candidate_retention_filter',
        'path': str(STRICT_PATH.relative_to(ROOT)),
        'exists': STRICT_PATH.exists(),
        'role': 'current strict no-promotion candidate filter and SOXLBacktest reference baseline',
    },
    {
        'source_label': 'bull_reentry_joint_target_contract',
        'path': str(TARGET_PATH.relative_to(ROOT)),
        'exists': TARGET_PATH.exists(),
        'role': 'fold-local bull reentry target contract for BC warm-start handoff',
    },
]
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True) + '\n', encoding='utf-8')
(CARD / 'selection.json').write_text(json.dumps(selection, indent=2, sort_keys=True) + '\n', encoding='utf-8')
(CARD / 'bc_warmstart_strict_gate.md').write_text(render_bc_warmstart_strict_gate(gate), encoding='utf-8')
with (CARD / 'source_manifest.csv').open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['source_label', 'path', 'exists', 'role'])
    writer.writeheader()
    writer.writerows(source_manifest)
summary

{'experiment': 'bull_reentry_bc_warmstart_gate_v1',
 'decision': 'bull_reentry_bc_warmstart_allowed_no_promotion',
 'bc_warm_start_allowed': True,
 'training_allowed_next': True,
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'ppo_allowed_next': False,
 'reference_baseline': 'soxl_backtest_policy_reimplementation_v1',
 'required_gates': {'required_wf_windows': ['wf_2020',
   'wf_2022',
   'wf_2023_2024'],
  'max_walkforward_drawdown_pct': 20.0,
  'reference_baseline': 'soxl_backtest_policy_reimplementation_v1',
  'required_best_return_pct': 213.143462},
 'required_next_gate': 'full_fixed_split_and_walkforward_retention_gate',
 'next_mechanism': 'wire_fold_local_bull_reentry_joint_targets_into_bc_warm_start_then_full_walkforward_gate',
 'source_target_contract': {'experiment': 'bull_reentry_joint_target_contract_v1',
  'decision': 'joint_target_contract_bull_reentry_gate_ready_for_training_handoff',
  'rows': 3,
  'bull_reentry_target_rows': 2,
  'eval_overlap_rows': 0

## Conclusion

Current status: BC warm-start training handoff is allowed, but promotion and actor retention remain blocked. The next model step must train from the fold-local bull reentry target contract and then rerun the full fixed-split and walk-forward retention gate with SOXLBacktest as the reference baseline, required windows `wf_2020`, `wf_2022`, `wf_2023_2024`, MaxDD below `20%`, and required best-return alignment `213.143462%`.